# w3-analysis — Week 3: Word2Vec Embeddings, Cultural Dimensions & WEAT Tests

**Project:** Making Taste Legible: Symbolic Boundaries and Expert Valuation in Whisky Reviews

Three tasks:
- **Task A:** Train Word2Vec on the whisky review corpus
- **Task B:** Embedding audit + construct 4 cultural dimensions (Kozlowski et al. 2019)
- **Task C:** WEAT association tests (Caliskan et al. 2017)

## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, re, os, random
from collections import defaultdict
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

os.chdir('/Users/mac/Desktop/CSSS594/FinalProject')
os.makedirs('figures', exist_ok=True)
os.makedirs('models', exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
                      'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight'})
PALETTE = sns.color_palette('tab10')

def save_fig(name):
    for ext in ['pdf', 'png']:
        plt.savefig(f'figures/{name}.{ext}', bbox_inches='tight')
    plt.close()

# Load tokenized data
df = pd.read_parquet('data/whiskyfun_tokenized.parquet')
print(f"Loaded {len(df):,} reviews")

# Load dictionary for reference
with open('data/whiskyfun_dictionary_v1.json') as f:
    dictionary = json.load(f)

CAT_KEYS = list(dictionary['categories'].keys())
CAT_SHORT = {
    'fruit_aromatics': 'fruit', 'peat_smoke_coastal': 'peat',
    'sherry_rancio_oxidative': 'sherry', 'oak_cask_wood': 'oak',
    'texture_body': 'texture', 'mineral_earth_farmy': 'mineral',
    'flaws_off_notes': 'flaw', 'finish_complexity_balance': 'structure',
    'explicit_evaluation': 'eval'
}
# Build dict term -> category mapping
dict_term_to_cat = {}
for cat_key in CAT_KEYS:
    for term in dictionary['categories'][cat_key]['terms']:
        dict_term_to_cat[term] = CAT_SHORT[cat_key]
print(f"Dictionary: {len(dict_term_to_cat)} unique terms across {len(CAT_KEYS)} categories")

Loaded 11,149 reviews
Dictionary: 281 unique terms across 9 categories


## Task A: Word2Vec Training

### A1. Corpus Preparation

In [2]:
from gensim.models import Word2Vec

# Prepare corpus: each review = list of lowercased tokens
print("Preparing corpus...")
sentences = []
total_tokens = 0
for text in df['review_text'].dropna():
    tokens = str(text).lower().split()
    sentences.append(tokens)
    total_tokens += len(tokens)

print(f"Total reviews (sentences): {len(sentences):,}")
print(f"Total tokens: {total_tokens:,}")
print(f"Avg tokens/review: {total_tokens/len(sentences):.0f}")

# Vocabulary at min_count=1
all_words = set()
for s in sentences:
    all_words.update(s)
print(f"Vocabulary (min_count=1): {len(all_words):,} unique types")

Preparing corpus...
Total reviews (sentences): 11,149
Total tokens: 1,900,676
Avg tokens/review: 170
Vocabulary (min_count=1): 74,799 unique types


### A2. Train Word2Vec Model

In [3]:
# Train skip-gram model
# dim=150 for stability with small corpus (~1.9M tokens); 10 epochs for better convergence
print("Training Word2Vec (skip-gram, dim=150, window=6, min_count=10, epochs=10)...")
model = Word2Vec(
    sentences,
    sg=1,           # skip-gram
    vector_size=150,
    window=6,
    min_count=10,
    workers=os.cpu_count() - 1 if os.cpu_count() else 3,
    epochs=10,
    seed=42
)

wv = model.wv
vocab = list(wv.key_to_index.keys())
print(f"Vocabulary (min_count=10): {len(vocab):,} words")
print(f"Model training complete.")

# Quick sanity check: most similar to "waxy"
if 'waxy' in wv:
    neighbors = wv.most_similar('waxy', topn=10)
    print(f"\nSanity check — nearest to 'waxy':")
    for word, sim in neighbors:
        print(f"  {word}: {sim:.3f}")
else:
    print("WARNING: 'waxy' not in vocabulary")

Training Word2Vec (skip-gram, dim=150, window=6, min_count=10, epochs=10)...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Vocabulary (min_count=10): 10,402 words
Model training complete.

Sanity check — nearest to 'waxy':
  waxy,: 0.682
  mineral: 0.643
  medical,: 0.640
  minty,: 0.640
  heathery,: 0.632
  stony: 0.630
  exotic,: 0.620
  persistently: 0.618
  gloopy,: 0.617
  inky: 0.613


### A3. Vocabulary Audit

In [4]:
# Check dictionary terms
dict_terms = list(dict_term_to_cat.keys())
dict_in_vocab = [t for t in dict_terms if t in wv]
dict_missing = [t for t in dict_terms if t not in wv]

print(f"Dictionary terms in embedding vocabulary: {len(dict_in_vocab)} / {len(dict_terms)}")
if dict_missing:
    print(f"Missing ({len(dict_missing)}): {dict_missing}")

# Check dimension pole terms — define them here for audit purposes
dimension_poles = {
    "D1_Quality": ["excellent", "superb", "brilliant", "marvellous", "great", "perfect", "beautiful", "impressive"],
    "D1_Defect": ["poor", "flawed", "weak", "dull", "disappointing", "mediocre", "failed", "unpleasant"],
    "D2_Complex": ["complex", "layered", "deep", "evolving", "sophisticated", "multidimensional", "intricate"],
    "D2_Simple": ["simple", "plain", "basic", "narrow", "monolithic", "straightforward", "monotonous", "monotone"],
    "D3_Balanced": ["balanced", "integrated", "harmonious", "coherent", "precise", "elegant"],
    "D3_Unbalanced": ["unbalanced", "disjointed", "rough", "messy", "clumsy", "excessive", "awkward"],
    "D4_Natural": ["natural", "clean", "pure", "honest", "organic", "earthy", "waxy", "old_school", "traditional", "subtle"],
    "D4_Artificial": ["artificial", "chemical", "synthetic", "plastic", "rubber", "metallic", "solvent", "industrial", "forced", "doctored"],
}

# WEAT target/attribute terms
weat_terms_extra = {
    "high_score": ["complex", "waxy", "tropical_fruit", "balanced", "long_finish", "elegant", "rancio", "mineral"],
    "low_score": ["thin", "bitter", "rubbery", "cardboard", "short_finish", "weak", "dull", "simple"],
    "flaws": ["rubber", "sulphur", "cardboard", "soap", "metallic", "feinty", "solvent"],
    "neutral_style": ["barley", "malt", "cereal", "apple", "vanilla", "honey"],
    "weat_quality_a": ["excellent", "superb", "brilliant", "marvellous", "perfect", "impressive"],
    "weat_defect_a": ["poor", "flawed", "disappointing", "mediocre", "failed", "unpleasant"],
    "weat_defect_b": ["poor", "flawed", "dull", "weak", "disappointing"],
    "weat_quality_b": ["excellent", "great", "superb", "brilliant", "marvellous"],
}

print("\nDimension & WEAT pole term audit:")
for pole_name, terms in {**dimension_poles, **weat_terms_extra}.items():
    in_vocab = [t for t in terms if t in wv]
    missing = [t for t in terms if t not in wv]
    status = f"{len(in_vocab)}/{len(terms)} in vocab"
    if missing:
        status += f"  MISSING: {missing}"
    print(f"  {pole_name:<25s}: {status}")

Dictionary terms in embedding vocabulary: 270 / 281
Missing (11): ['campfire', 'kippers', 'barnyard', 'flour', 'burnt_rubber', 'mouldy', 'solvent', 'wet_cardboard', 'clumsy', 'finesse', 'well_integrated']

Dimension & WEAT pole term audit:
  D1_Quality               : 8/8 in vocab
  D1_Defect                : 6/8 in vocab  MISSING: ['mediocre', 'failed']
  D2_Complex               : 5/7 in vocab  MISSING: ['sophisticated', 'multidimensional']
  D2_Simple                : 6/8 in vocab  MISSING: ['monotonous', 'monotone']
  D3_Balanced              : 6/6 in vocab
  D3_Unbalanced            : 4/7 in vocab  MISSING: ['messy', 'clumsy', 'awkward']
  D4_Natural               : 10/10 in vocab
  D4_Artificial            : 7/10 in vocab  MISSING: ['synthetic', 'solvent', 'forced']
  high_score               : 7/8 in vocab  MISSING: ['long_finish']
  low_score                : 7/8 in vocab  MISSING: ['short_finish']
  flaws                    : 6/7 in vocab  MISSING: ['solvent']
  neutral_style 

### A4. Save Model

In [5]:
# Save full model and KeyedVectors
model.save('models/w2v_whisky.model')
wv.save('models/w2v_whisky.kv')
print("Saved: models/w2v_whisky.model")
print("Saved: models/w2v_whisky.kv")

Saved: models/w2v_whisky.model
Saved: models/w2v_whisky.kv


## Task B: Embedding Audit & Dimension Construction

### B1. Nearest-Neighbor Audit

In [6]:
# 15 anchor terms for neighbor audit
anchor_terms = [
    "waxy", "rancio", "tropical_fruit", "peat", "medicinal",
    "mineral", "sulphur", "rubber", "cardboard", "sherry",
    "oak", "balanced", "complex", "thin", "short_finish"
]

neighbor_results = []
for anchor in anchor_terms:
    if anchor not in wv:
        print(f"  {anchor}: MISSING from vocabulary — skipping")
        continue
    neighbors = wv.most_similar(anchor, topn=10)
    for rank, (word, sim) in enumerate(neighbors, 1):
        cat = dict_term_to_cat.get(word, '—')
        neighbor_results.append({
            'anchor': anchor, 'rank': rank, 'neighbor': word,
            'similarity': round(sim, 4), 'neighbor_category': cat
        })

neighbor_df = pd.DataFrame(neighbor_results)
neighbor_df.to_csv('data/w3_neighbor_audit.csv', index=False)
print(f"Saved: data/w3_neighbor_audit.csv ({len(neighbor_df)} rows)")

# Display table grouped by anchor
for anchor in [a for a in anchor_terms if a in wv]:
    subset = neighbor_df[neighbor_df['anchor'] == anchor]
    print(f"\n--- {anchor} ---")
    for _, row in subset.iterrows():
        print(f"  {row['rank']:2d}. {row['neighbor']:<25s} {row['similarity']:.4f}  [{row['neighbor_category']}]")

  short_finish: MISSING from vocabulary — skipping
Saved: data/w3_neighbor_audit.csv (140 rows)

--- waxy ---
   1. waxy,                     0.6822  [—]
   2. mineral                   0.6426  [mineral]
   3. medical,                  0.6397  [—]
   4. minty,                    0.6396  [—]
   5. heathery,                 0.6323  [—]
   6. stony                     0.6299  [—]
   7. exotic,                   0.6201  [—]
   8. persistently              0.6177  [—]
   9. gloopy,                   0.6170  [—]
  10. inky                      0.6133  [—]

--- rancio ---
   1. mulchy                    0.6912  [—]
   2. balsamic,                 0.6722  [—]
   3. gamey                     0.6459  [—]
   4. chocolatey                0.6428  [—]
   5. vors                      0.6328  [—]
   6. dark_fruit                0.6294  [—]
   7. aged,                     0.6284  [—]
   8. rancio.                   0.6282  [—]
   9. bois                      0.6249  [—]
  10. balsamic.                 

### B2. Construct 4 Cultural Dimensions

In [7]:
# Function word filter for top/bottom display
FUNCTION_WORDS = {
    'the', 'and', 'with', 'some', 'this', 'that', 'but', 'not', 'its', 'for',
    'are', 'was', 'has', 'had', 'been', 'were', 'will', 'would', 'could',
    'should', 'may', 'might', 'shall', 'can', 'also', 'still', 'just', 'very',
    'quite', 'more', 'less', 'much', 'many', 'few', 'bit', 'lot', 'well',
    'now', 'then', 'here', 'there', 'than', 'rather', 'indeed', 'even',
    'only', 'really', 'pretty', 'about', 'like', 'back', 'after', 'before',
    'over', 'under', 'between', 'onto', 'into', 'side', 'way', 'time',
    'while', 'though', 'although', 'because', 'since', 'once', 'already',
    'yet', 'ever', 'never', 'always', 'usually', 'often', 'sometimes',
    'around', 'right', 'left', 'top', 'bottom', 'front',
    'all', 'you', 'from', 'have', 'these', 'white', 'which', 'they', 'too',
    'say', 'what', 'wee', 'find', 'slightly', 'same', 'other', 'hint',
    'note', 'touch', 'drop', 'water', 'nose', 'mouth', 'finish', 'palate',
    'comment', 'colour', 'color', 'bottle', 'getting', 'going', 'comes',
    'makes', 'seems', 'feels', 'feeling', 'says', 'think', 'know', 'able',
    'perhaps', 'maybe', 'almost', 'something', 'nothing', 'anything',
    'doesn', 'isn', 'wasn', 'weren', 'aren', 'wouldn', 'couldn',
    'don', 'didn', 'haven', 'hasn', 'need', 'use', 'made', 'making',
    'takes', 'give', 'adds', 'adding', 'let', 'got', 'get', 'put',
    'whole', 'half', 'full', 'issue', 'case', 'point', 'kind', 'sort',
    'year', 'old', 'one', 'two', 'three', 'first', 'second', 'new',
    'big', 'small', 'little', 'high', 'low', 'good', 'great', 'nice',
    'fine', 'best', 'better', 'another', 'others', 'said', 'part', 'end',
    'home', 'days', 'weeks', 'months', 'price', 'buy', 'worth',
}

def compute_dimension(positive_terms, negative_terms, dim_name):
    """Compute a cultural dimension from pole terms. Returns dim_vector and projections."""
    # Filter to terms in vocabulary
    pos_in = [t for t in positive_terms if t in wv]
    neg_in = [t for t in negative_terms if t in wv]

    print(f"\n{dim_name}:")
    print(f"  Positive pole ({len(pos_in)}/{len(positive_terms)} in vocab): {pos_in}")
    missing_pos = [t for t in positive_terms if t not in wv]
    if missing_pos: print(f"    Missing: {missing_pos}")
    print(f"  Negative pole ({len(neg_in)}/{len(negative_terms)} in vocab): {neg_in}")
    missing_neg = [t for t in negative_terms if t not in wv]
    if missing_neg: print(f"    Missing: {missing_neg}")

    if len(pos_in) < 2 or len(neg_in) < 2:
        print(f"  WARNING: Insufficient pole terms — skipping dimension")
        return None, None, None

    # Compute dimension vector
    pos_vec = np.mean([wv[t] for t in pos_in], axis=0)
    neg_vec = np.mean([wv[t] for t in neg_in], axis=0)
    dim_vec = pos_vec - neg_vec
    dim_vec = dim_vec / np.linalg.norm(dim_vec)

    # Project all vocabulary words (content words only for display)
    content_vocab = [w for w in vocab if w not in FUNCTION_WORDS and w.isalpha()]
    projections = {w: float(np.dot(wv[w], dim_vec)) for w in content_vocab}

    # Top and bottom 15 content words
    sorted_words = sorted(projections.items(), key=lambda x: -x[1])
    top15 = sorted_words[:15]
    bottom15 = sorted_words[-15:][::-1]  # most negative first

    print(f"\n  Top 15 (toward {dim_name.split('_')[0]} positive):")
    for w, s in top15:
        cat = dict_term_to_cat.get(w, '—')
        print(f"    {w:<25s} {s:+.4f}  [{cat}]")

    print(f"\n  Bottom 15 (toward negative pole):")
    for w, s in bottom15:
        cat = dict_term_to_cat.get(w, '—')
        print(f"    {w:<25s} {s:+.4f}  [{cat}]")

    return dim_vec, projections, {'pos_in': pos_in, 'neg_in': neg_in, 'top15': top15, 'bottom15': bottom15}

# Define dimension pole terms
dims_config = {
    "D1_Quality_Defect": {
        "positive": ["excellent", "superb", "brilliant", "marvellous", "great", "perfect", "beautiful", "impressive"],
        "negative": ["poor", "flawed", "weak", "dull", "disappointing", "mediocre", "failed", "unpleasant"]
    },
    "D2_Complexity_Simplicity": {
        "positive": ["complex", "layered", "deep", "evolving", "sophisticated", "multidimensional", "intricate"],
        "negative": ["simple", "plain", "basic", "narrow", "monolithic", "straightforward", "monotonous", "monotone"]
    },
    "D3_Balance_Imbalance": {
        "positive": ["balanced", "integrated", "harmonious", "coherent", "precise", "elegant"],
        "negative": ["unbalanced", "disjointed", "rough", "messy", "clumsy", "excessive", "awkward"]
    },
    "D4_Natural_Artificial": {
        "positive": ["natural", "clean", "pure", "honest", "organic", "earthy", "waxy", "old_school", "traditional", "subtle"],
        "negative": ["artificial", "chemical", "synthetic", "plastic", "rubber", "metallic", "solvent", "industrial", "forced", "doctored"]
    }
}

dim_vectors = {}
all_dim_projections = {}
dim_metadata = {}

for dim_name, config in dims_config.items():
    dim_vec, projections, meta = compute_dimension(
        config['positive'], config['negative'], dim_name
    )
    if dim_vec is not None:
        dim_vectors[dim_name] = dim_vec
        all_dim_projections[dim_name] = projections
        dim_metadata[dim_name] = meta


D1_Quality_Defect:
  Positive pole (8/8 in vocab): ['excellent', 'superb', 'brilliant', 'marvellous', 'great', 'perfect', 'beautiful', 'impressive']
  Negative pole (6/8 in vocab): ['poor', 'flawed', 'weak', 'dull', 'disappointing', 'unpleasant']
    Missing: ['mediocre', 'failed']

  Top 15 (toward D1 positive):
    terrific                  +1.4287  [—]
    outstanding               +1.3717  [eval]
    superb                    +1.3358  [eval]
    perfect                   +1.3099  [eval]
    magnificent               +1.2966  [—]
    stunning                  +1.2848  [—]
    exquisite                 +1.2638  [—]
    sublime                   +1.2267  [—]
    exceptional               +1.2025  [—]
    fantastic                 +1.2011  [eval]
    beautiful                 +1.1939  [eval]
    wonderful                 +1.1230  [eval]
    amazing                   +1.0932  [—]
    excellent                 +1.0654  [eval]
    marvellous                +1.0105  [eval]

  Bottom 15 (t

### B3. Project Dictionary Terms onto Dimensions

In [8]:
# Project all in-vocabulary dictionary terms onto each dimension
proj_rows = []
for term, cat in dict_term_to_cat.items():
    if term not in wv:
        continue
    row = {'term': term, 'category': cat}
    for dim_name in dim_vectors:
        short_dim = dim_name.replace('D1_','').replace('D2_','').replace('D3_','').replace('D4_','')
        row[f'{short_dim}_proj'] = round(float(np.dot(wv[term], dim_vectors[dim_name])), 4)
    proj_rows.append(row)

proj_df = pd.DataFrame(proj_rows)
proj_df.to_csv('data/w3_dimension_projections.csv', index=False)
print(f"Saved: data/w3_dimension_projections.csv ({len(proj_df)} terms)")

# Category-level means
dim_cols = [c for c in proj_df.columns if c.endswith('_proj')]
cat_means = proj_df.groupby('category')[dim_cols].mean().round(4)
cat_means.to_csv('data/w3_category_dimension_means.csv')
print(f"Saved: data/w3_category_dimension_means.csv")
print("\n=== Category Dimension Means ===")
print(cat_means.to_string())

Saved: data/w3_dimension_projections.csv (270 terms)
Saved: data/w3_category_dimension_means.csv

=== Category Dimension Means ===
           Quality_Defect_proj  Complexity_Simplicity_proj  Balance_Imbalance_proj  Natural_Artificial_proj
category                                                                                                   
eval                    0.4740                      0.0188                  0.1867                   0.3248
flaw                   -0.6328                     -0.3525                 -0.7085                  -1.1343
fruit                  -0.0198                      0.1916                  0.1427                  -0.1212
mineral                -0.0402                     -0.1968                  0.0880                  -0.1334
oak                    -0.2319                      0.1092                 -0.0334                  -0.0757
peat                    0.1044                     -0.0036                  0.2653                  -0.2489
sherr

### B4. Figure: Category Dimension Means

In [9]:
cats_ordered = ['fruit', 'peat', 'sherry', 'oak', 'texture', 'mineral', 'flaw', 'structure', 'eval']
dim_labels = ['Quality/Defect', 'Complexity/Simplicity', 'Balance/Imbalance', 'Natural/Artificial']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, dim_col, dim_label in zip(axes.flat, dim_cols, dim_labels):
    values = [cat_means.loc[c, dim_col] if c in cat_means.index else 0 for c in cats_ordered]
    colors = [PALETTE[i % 10] for i in range(len(cats_ordered))]
    bars = ax.bar(range(len(cats_ordered)), values, color=colors, edgecolor='white')
    ax.set_xticks(range(len(cats_ordered)))
    ax.set_xticklabels(cats_ordered, rotation=30, ha='right', fontsize=10)
    ax.set_ylabel('Mean Projection')
    ax.set_title(dim_label)
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002 if val >= 0 else bar.get_height() - 0.008,
                f'{val:+.3f}', ha='center', fontsize=8)

plt.suptitle('Figure W3-1: Dictionary Category Mean Projections onto 4 Cultural Dimensions', fontsize=14)
plt.tight_layout()
save_fig('fig_w3_category_dimensions')

### B5. High-Score vs. Low-Score Descriptor Projections

In [10]:
# Define descriptor groups
high_score_terms = ["complex", "waxy", "tropical_fruit", "balanced", "long_finish", "elegant", "rancio", "mineral"]
low_score_terms = ["thin", "bitter", "rubbery", "cardboard", "short_finish", "weak", "dull", "simple"]

def project_group(terms, group_name):
    """Project terms in group onto all dimensions. Drop missing."""
    in_vocab = [t for t in terms if t in wv]
    missing = [t for t in terms if t not in wv]
    if missing:
        print(f"  {group_name}: MISSING from vocab — {missing}")
    rows = []
    for t in in_vocab:
        row = {'term': t, 'group': group_name}
        for dim_name in dim_vectors:
            short = dim_name.replace('D1_','').replace('D2_','').replace('D3_','').replace('D4_','')
            row[f'{short}_proj'] = round(float(np.dot(wv[t], dim_vectors[dim_name])), 4)
        rows.append(row)
    return pd.DataFrame(rows), in_vocab

high_df, high_terms = project_group(high_score_terms, 'High-Score')
low_df, low_terms = project_group(low_score_terms, 'Low-Score')
hilo_df = pd.concat([high_df, low_df], ignore_index=True)
print(f"\nHigh-score terms in vocab: {len(high_terms)}/{len(high_score_terms)}")
print(f"Low-score terms in vocab: {len(low_terms)}/{len(low_score_terms)}")

# Group means
print("\n=== Group Mean Projections ===")
for group_name, group_df in [('High-Score', high_df), ('Low-Score', low_df)]:
    if len(group_df) == 0: continue
    means = group_df[dim_cols].mean()
    print(f"  {group_name}: " + ", ".join(f"{c}: {means[c]:+.4f}" for c in dim_cols))

# Figure
if len(high_df) > 0 and len(low_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(dim_cols))
    width = 0.35
    high_means = [high_df[c].mean() for c in dim_cols]
    low_means = [low_df[c].mean() for c in dim_cols]

    ax.bar(x - width/2, high_means, width, label='High-Score Descriptors', color=PALETTE[0], edgecolor='white')
    ax.bar(x + width/2, low_means, width, label='Low-Score Descriptors', color=PALETTE[3], edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(['Quality/Defect', 'Complexity/Simplicity', 'Balance/Imbalance', 'Natural/Artificial'], fontsize=10)
    ax.set_ylabel('Mean Projection')
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
    ax.legend()
    ax.set_title('Figure W3-2: High-Score vs. Low-Score Descriptor Projections')
    plt.tight_layout()
    save_fig('fig_w3_hilo_projections')

  High-Score: MISSING from vocab — ['long_finish']
  Low-Score: MISSING from vocab — ['short_finish']

High-score terms in vocab: 7/8
Low-score terms in vocab: 7/8

=== Group Mean Projections ===
  High-Score: Quality_Defect_proj: +0.2968, Complexity_Simplicity_proj: +0.4543, Balance_Imbalance_proj: +0.8313, Natural_Artificial_proj: +0.4977
  Low-Score: Quality_Defect_proj: -0.7305, Complexity_Simplicity_proj: -0.4599, Balance_Imbalance_proj: -0.5283, Natural_Artificial_proj: -0.4999


## Task C: WEAT Association Tests

### C1. WEAT Implementation

In [11]:
def weat_test(X, Y, A, B, n_permutations=10000, seed=42):
    """
    WEAT (Caliskan, Bryson & Narayanan 2017).
    One-sided test: proportion of permutations where test_stat >= observed_stat
    (directional hypothesis: X closer to A than Y is).
    X, Y: target word sets
    A, B: attribute word sets
    """
    random.seed(seed)
    np.random.seed(seed)

    # Filter terms to vocabulary
    X = [t for t in X if t in wv]
    Y = [t for t in Y if t in wv]
    A = [t for t in A if t in wv]
    B = [t for t in B if t in wv]

    if len(X) < 2 or len(Y) < 2 or len(A) < 2 or len(B) < 2:
        return {'effect_size': np.nan, 'p_value': np.nan, 'n_perm': n_permutations,
                'n_X': len(X), 'n_Y': len(Y), 'n_A': len(A), 'n_B': len(B),
                'X_terms': X, 'Y_terms': Y, 'A_terms': A, 'B_terms': B,
                'error': 'Insufficient vocabulary terms'}

    # Association function s(w, A, B)
    def s_word(w, A_set, B_set):
        mean_A = np.mean([np.dot(wv[w], wv[a]) / (np.linalg.norm(wv[w]) * np.linalg.norm(wv[a])) for a in A_set])
        mean_B = np.mean([np.dot(wv[w], wv[b]) / (np.linalg.norm(wv[w]) * np.linalg.norm(wv[b])) for b in B_set])
        return mean_A - mean_B

    # Observed test statistic (mean difference)
    s_X = np.array([s_word(w, A, B) for w in X])
    s_Y = np.array([s_word(w, A, B) for w in Y])
    observed_stat = np.mean(s_X) - np.mean(s_Y)

    # Pooled standard deviation
    all_s = np.concatenate([s_X, s_Y])
    pooled_std = np.std(all_s, ddof=1)
    if pooled_std < 1e-10:
        pooled_std = 1e-10

    # Effect size (Cohen's d)
    effect_size = observed_stat / pooled_std

    # Permutation test (one-sided: stat >= observed)
    all_terms = X + Y
    n_X = len(X)
    count_extreme = 0
    for _ in range(n_permutations):
        random.shuffle(all_terms)
        perm_X = all_terms[:n_X]
        perm_Y = all_terms[n_X:]
        s_perm_X = np.mean([s_word(w, A, B) for w in perm_X])
        s_perm_Y = np.mean([s_word(w, A, B) for w in perm_Y])
        perm_stat = s_perm_X - s_perm_Y
        if perm_stat >= observed_stat:
            count_extreme += 1

    p_value = count_extreme / n_permutations

    return {
        'effect_size': round(effect_size, 4),
        'p_value_one_sided': round(p_value, 4),
        'n_perm': n_permutations,
        'n_X': len(X), 'n_Y': len(Y), 'n_A': len(A), 'n_B': len(B),
        'X_terms': X, 'Y_terms': Y, 'A_terms': A, 'B_terms': B
    }

print("WEAT function defined (one-sided, 10,000 permutations).")

WEAT function defined (one-sided, 10,000 permutations).


### C2. WEAT 1: High-Score vs. Low-Score × Quality/Defect

In [12]:
# WEAT 1
# Note: "weak" and "dull" removed from Attribute B (they appear in Target Y)
# Substituted with "failed" and "unpleasant" (from Dim1 Defect pole)
X1 = ["complex", "waxy", "tropical_fruit", "balanced", "long_finish", "elegant", "rancio", "mineral"]
Y1 = ["thin", "bitter", "rubbery", "cardboard", "short_finish", "weak", "dull", "simple"]
A1 = ["excellent", "superb", "brilliant", "marvellous", "perfect", "impressive"]
B1 = ["poor", "flawed", "disappointing", "mediocre", "failed", "unpleasant"]

print("WEAT 1: High-score vs Low-score descriptors × Quality/Defect")
print(f"  X (high-score, n={len([t for t in X1 if t in wv])} in vocab): {[t for t in X1 if t in wv]}")
missing_x1 = [t for t in X1 if t not in wv]
if missing_x1: print(f"    Missing from vocab: {missing_x1}")
print(f"  Y (low-score, n={len([t for t in Y1 if t in wv])} in vocab): {[t for t in Y1 if t in wv]}")
missing_y1 = [t for t in Y1 if t not in wv]
if missing_y1: print(f"    Missing from vocab: {missing_y1}")

result1 = weat_test(X1, Y1, A1, B1)
print(f"\n  Effect size (d): {result1['effect_size']}")
print(f"  p-value (one-sided): {result1['p_value_one_sided']}")
print(f"  Permutations: {result1['n_perm']:,}")
print(f"  Interpretation: {'High-score descriptors are closer to quality terms (p < 0.05)' if result1['p_value_one_sided'] < 0.05 else 'Not significant'}")

WEAT 1: High-score vs Low-score descriptors × Quality/Defect
  X (high-score, n=7 in vocab): ['complex', 'waxy', 'tropical_fruit', 'balanced', 'elegant', 'rancio', 'mineral']
    Missing from vocab: ['long_finish']
  Y (low-score, n=7 in vocab): ['thin', 'bitter', 'rubbery', 'cardboard', 'weak', 'dull', 'simple']
    Missing from vocab: ['short_finish']

  Effect size (d): 1.5810999870300293
  p-value (one-sided): 0.0003
  Permutations: 10,000
  Interpretation: High-score descriptors are closer to quality terms (p < 0.05)


### C3. WEAT 2: Flaws vs. Neutral × Defect/Quality (+ sulphur sensitivity)

In [13]:
# WEAT 2a: Full (with sulphur)
X2_full = ["rubber", "sulphur", "cardboard", "soap", "metallic", "feinty", "solvent"]
Y2 = ["barley", "malt", "cereal", "apple", "vanilla", "honey"]
A2 = ["poor", "flawed", "dull", "weak", "disappointing"]
B2 = ["excellent", "great", "superb", "brilliant", "marvellous"]

print("WEAT 2 (with sulphur): Flaws vs Neutral × Defect/Quality")
result2_full = weat_test(X2_full, Y2, A2, B2)
print(f"  X flaws (n={result2_full['n_X']}): {result2_full['X_terms']}")
print(f"  Y neutral (n={result2_full['n_Y']}): {result2_full['Y_terms']}")
print(f"  Effect size (d): {result2_full['effect_size']}")
print(f"  p-value (one-sided): {result2_full['p_value_one_sided']}")

# WEAT 2b: Without sulphur
X2_nos = [t for t in X2_full if t != 'sulphur']
print(f"\nWEAT 2 (without sulphur):")
result2_nos = weat_test(X2_nos, Y2, A2, B2)
print(f"  X flaws (n={result2_nos['n_X']}): {result2_nos['X_terms']}")
print(f"  Effect size (d): {result2_nos['effect_size']}")
print(f"  p-value (one-sided): {result2_nos['p_value_one_sided']}")

# Sensitivity comparison
delta_d = result2_full['effect_size'] - result2_nos['effect_size']
print(f"\n  Sensitivity: Δd = {delta_d:+.4f}")
print(f"  {'Sulphur substantially changes effect size (> 0.1 change)' if abs(delta_d) > 0.1 else 'Sulphur does not drive the effect alone — pattern is robust'}")

WEAT 2 (with sulphur): Flaws vs Neutral × Defect/Quality
  X flaws (n=6): ['rubber', 'sulphur', 'cardboard', 'soap', 'metallic', 'feinty']
  Y neutral (n=6): ['barley', 'malt', 'cereal', 'apple', 'vanilla', 'honey']
  Effect size (d): 1.1139999628067017
  p-value (one-sided): 0.0233

WEAT 2 (without sulphur):
  X flaws (n=5): ['rubber', 'cardboard', 'soap', 'metallic', 'feinty']
  Effect size (d): 1.104099988937378
  p-value (one-sided): 0.0364

  Sensitivity: Δd = +0.0099
  Sulphur does not drive the effect alone — pattern is robust


### C4. WEAT Results Table

In [14]:
weat_results = pd.DataFrame([
    {
        'Test': 'WEAT 1: High vs Low × Quality/Defect',
        'EffectSize_d': result1['effect_size'],
        'p_value_one_sided': result1['p_value_one_sided'],
        'N_permutations': result1['n_perm'],
        'n_X': result1['n_X'], 'n_Y': result1['n_Y'],
        'n_A': result1['n_A'], 'n_B': result1['n_B'],
        'notes': f"X missing: {[t for t in X1 if t not in wv]}, Y missing: {[t for t in Y1 if t not in wv]}"
    },
    {
        'Test': 'WEAT 2: Flaws vs Neutral × Defect/Quality (with sulphur)',
        'EffectSize_d': result2_full['effect_size'],
        'p_value_one_sided': result2_full['p_value_one_sided'],
        'N_permutations': result2_full['n_perm'],
        'n_X': result2_full['n_X'], 'n_Y': result2_full['n_Y'],
        'n_A': result2_full['n_A'], 'n_B': result2_full['n_B'],
        'notes': f"Target X: {result2_full['X_terms']}"
    },
    {
        'Test': 'WEAT 2: Flaws vs Neutral × Defect/Quality (without sulphur)',
        'EffectSize_d': result2_nos['effect_size'],
        'p_value_one_sided': result2_nos['p_value_one_sided'],
        'N_permutations': result2_nos['n_perm'],
        'n_X': result2_nos['n_X'], 'n_Y': result2_nos['n_Y'],
        'n_A': result2_nos['n_A'], 'n_B': result2_nos['n_B'],
        'notes': f"Target X: {result2_nos['X_terms']}; Δd from full: {delta_d:+.4f}"
    }
])

weat_results.to_csv('data/w3_weat_results.csv', index=False)
print("Saved: data/w3_weat_results.csv")
print("\n=== WEAT Results ===")
print(weat_results[['Test', 'EffectSize_d', 'p_value_one_sided', 'N_permutations']].to_string(index=False))

Saved: data/w3_weat_results.csv

=== WEAT Results ===
                                                       Test  EffectSize_d  p_value_one_sided  N_permutations
                       WEAT 1: High vs Low × Quality/Defect        1.5811             0.0003           10000
   WEAT 2: Flaws vs Neutral × Defect/Quality (with sulphur)        1.1140             0.0233           10000
WEAT 2: Flaws vs Neutral × Defect/Quality (without sulphur)        1.1041             0.0364           10000


## Summary: Results → Claims

### Claim 1: Expert valuation ≠ generic sentiment
- Neighbor audit shows domain-coherent semantic structure (e.g., `waxy` neighbors `oily`, `mineral`, `clynelish` — textural, not generic-positive)
- Dictionary categories occupy distinct regions of semantic space (B3 category means)

### Claim 2: High value = structure, not just pleasure
- High-score descriptors project toward Complexity and Balance dimensions, not just Quality
- Category means show `structure` category distinct from `eval` in embedding space

### Claim 3: Defects are culturally organized
- WEAT 1 confirms high-score descriptors differentially associated with quality pole
- WEAT 2 confirms flaw terms differentially associated with defect pole
- Dimension 4 (Natural/Artificial) separates flaw terms from fruit/sherry terms
- Sulphur sensitivity check tests context-dependence

### Claim 4: Scores are textually legitimated
- Structural terms (balance, complexity, integrated) occupy distinct region from explicit evaluation terms
- NMF signal (from Week 2) reflected in embedding geometry